# Module 1: Panel Data Preparation

## Learning Objectives

By completing this notebook, you will:
1. Convert wide and long format data to panel structure
2. Create and manage MultiIndex DataFrames for panel data
3. Handle unbalanced panels and missing observations
4. Reshape panel data between wide and long formats
5. Validate panel structure and identify data quality issues

## Prerequisites

- Understanding of pandas DataFrames
- Basic knowledge of data structures (long vs wide format)
- Familiarity with indexing and slicing

## Estimated Time

60-75 minutes

---

## Setup

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

# Set random seed
np.random.seed(42)

print("✅ Setup complete")

In [ ]:
section_divider("1. Panel Data Formats")

In [ ]:
learning_objectives(["Convert wide and long format data to panel structure", "Create and manage MultiIndex DataFrames for panel data", "Handle unbalanced panels and missing observations", "Reshape panel data between wide and long formats", "Validate panel structure and identify data quality issues"])

In [ ]:
apply_course_theme()
apply_plot_theme()

## 1. Panel Data Formats

### What is Panel Data?

**Panel data** (longitudinal data) tracks multiple entities over time:
- **Cross-sectional dimension:** Different entities (firms, countries, individuals)
- **Time dimension:** Multiple periods for each entity

### Three Common Formats

**1. Long Format (Stacked):**
```
entity  time  var1  var2
  A     2020   10    100
  A     2021   12    110
  B     2020   15    90
  B     2021   18    95
```

**2. Wide Format:**
```
entity  var1_2020  var1_2021  var2_2020  var2_2021
  A        10         12         100        110
  B        15         18          90         95
```

**3. MultiIndex (Hierarchical):**
```
entity time      var1  var2
  A    2020       10    100
       2021       12    110
  B    2020       15     90
       2021       18     95
```

For panel regression, we typically use **long format with MultiIndex**.

In [ ]:
section_divider("2. Creating Panel Data from Scratch")

## 2. Creating Panel Data from Scratch

Let's generate synthetic state-level economic data to demonstrate panel structure.

In [ ]:
# Generate state-level panel data
states = ['CA', 'TX', 'NY', 'FL', 'IL', 'PA', 'OH', 'GA', 'NC', 'MI']
years = np.arange(2010, 2024)
n_states = len(states)
n_years = len(years)

# Create all combinations (balanced panel)
panel_index = pd.MultiIndex.from_product(
    [states, years],
    names=['state', 'year']
)

# State characteristics (time-invariant)
state_base_gdp = np.random.randn(n_states) * 200 + 500
state_growth_rate = np.random.uniform(0.01, 0.04, n_states)

# Generate time-varying variables
data = []
for i, state in enumerate(states):
    for t, year in enumerate(years):
        # GDP with trend and random shock
        gdp = state_base_gdp[i] * (1 + state_growth_rate[i]) ** t + np.random.randn() * 20
        
        # Unemployment rate (mean-reverting)
        base_unemp = 5.0 + np.random.randn() * 0.5
        unemployment = base_unemp + np.sin(t / 2) * 1.5 + np.random.randn() * 0.3
        unemployment = max(2.0, min(12.0, unemployment))  # Constrain to realistic range
        
        # Tax rate (slowly changing)
        base_tax = 5.0 + i * 0.5  # States differ in tax rates
        tax_rate = base_tax + np.random.randn() * 0.2
        
        # Population (growing)
        base_pop = 5 + i * 2
        population = base_pop * (1.01 ** t) + np.random.randn() * 0.1
        
        data.append({
            'state': state,
            'year': year,
            'gdp': gdp,
            'unemployment': unemployment,
            'tax_rate': tax_rate,
            'population': population
        })

# Create DataFrame
df_long = pd.DataFrame(data)

print(f"Panel data created:")
print(f"  States (N): {n_states}")
print(f"  Years (T): {n_years}")
print(f"  Total observations: {len(df_long)}")
print(f"\nFirst 15 rows:")
print(df_long.head(15))

In [ ]:
section_divider("3. Converting to MultiIndex Panel Structure")

## 3. Converting to MultiIndex Panel Structure

For panel regression libraries (linearmodels, statsmodels), we need a **MultiIndex** with entity and time.

In [ ]:
# Set MultiIndex
df_panel = df_long.set_index(['state', 'year'])

print("Panel DataFrame with MultiIndex:")
print(df_panel.head(10))
print(f"\nIndex names: {df_panel.index.names}")
print(f"Index levels: {df_panel.index.nlevels}")
print(f"\nData types:")
print(df_panel.dtypes)

### Accessing Data with MultiIndex

MultiIndex enables powerful slicing operations:

In [ ]:
# Access single state (all years)
print("California data (all years):")
print(df_panel.loc['CA'].head())

# Access single year (all states)
print("\nYear 2020 data (all states):")
print(df_panel.xs(2020, level='year').head())

# Access specific state-year
print("\nCalifornia in 2020:")
print(df_panel.loc[('CA', 2020)])

# Access multiple states
print("\nCA and TX data (first 5 rows each):")
print(df_panel.loc[['CA', 'TX']].head(10))

In [ ]:
section_divider("4. Wide Format to Long Format Conversion")

## 4. Wide Format to Long Format Conversion

Data often comes in wide format. Let's practice converting it to long format.

In [ ]:
# Create wide format from our panel data
df_wide = df_panel.reset_index().pivot(
    index='state',
    columns='year',
    values=['gdp', 'unemployment']
)

# Flatten column names
df_wide.columns = ['_'.join(map(str, col)) for col in df_wide.columns]
df_wide = df_wide.reset_index()

print("Wide format data (GDP columns):")
print(df_wide[['state'] + [col for col in df_wide.columns if 'gdp' in col][:5]].head())

### Convert Wide to Long

Use `pd.melt()` or `pd.wide_to_long()` to convert wide format to long format.

In [ ]:
# Method 1: Using pd.melt
def wide_to_long_melt(df_wide, id_vars, value_vars_pattern):
    """
    Convert wide format to long using pd.melt.
    
    Parameters
    ----------
    df_wide : pd.DataFrame
        Wide format data
    id_vars : list
        Columns that identify entities
    value_vars_pattern : str
        Pattern to match value columns
    
    Returns
    -------
    pd.DataFrame
        Long format data
    """
    # Get GDP columns
    gdp_cols = [col for col in df_wide.columns if 'gdp' in col]
    unemp_cols = [col for col in df_wide.columns if 'unemployment' in col]
    
    # Melt GDP
    df_gdp = df_wide[id_vars + gdp_cols].melt(
        id_vars=id_vars,
        var_name='year',
        value_name='gdp'
    )
    df_gdp['year'] = df_gdp['year'].str.extract(r'(\d{4})').astype(int)
    
    # Melt unemployment
    df_unemp = df_wide[id_vars + unemp_cols].melt(
        id_vars=id_vars,
        var_name='year',
        value_name='unemployment'
    )
    df_unemp['year'] = df_unemp['year'].str.extract(r'(\d{4})').astype(int)
    
    # Merge
    df_long_reconstructed = df_gdp.merge(df_unemp, on=['state', 'year'])
    
    return df_long_reconstructed

df_long_reconstructed = wide_to_long_melt(df_wide, ['state'], 'gdp')

print("Reconstructed long format:")
print(df_long_reconstructed.head(10))
print(f"\nShape: {df_long_reconstructed.shape}")

In [ ]:
section_divider("5. Handling Unbalanced Panels")

## 5. Handling Unbalanced Panels

Real-world panel data often has **missing observations** (unbalanced panel).

In [ ]:
# Create unbalanced panel by randomly removing observations
df_unbalanced = df_long.copy()
np.random.seed(123)
missing_idx = np.random.choice(df_unbalanced.index, size=30, replace=False)
df_unbalanced = df_unbalanced.drop(missing_idx)

print(f"Original observations: {len(df_long)}")
print(f"Unbalanced panel observations: {len(df_unbalanced)}")
print(f"Missing observations: {len(df_long) - len(df_unbalanced)}")

# Check observations per state
obs_per_state = df_unbalanced.groupby('state').size()
print(f"\nObservations per state:")
print(obs_per_state)
print(f"\nMin: {obs_per_state.min()}, Max: {obs_per_state.max()}")

### Identify Missing Patterns

Understanding missing data patterns is crucial for choosing the right panel estimator.

In [ ]:
def analyze_panel_balance(df, entity_col, time_col):
    """
    Analyze panel balance and missing data patterns.
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
    time_col : str
    
    Returns
    -------
    dict
        Balance statistics
    """
    # Count observations per entity
    obs_per_entity = df.groupby(entity_col).size()
    
    # Expected observations (max time periods)
    n_entities = df[entity_col].nunique()
    n_periods = df[time_col].nunique()
    max_obs = n_entities * n_periods
    actual_obs = len(df)
    
    # Balance ratio
    balance_ratio = actual_obs / max_obs
    
    # Check if perfectly balanced
    is_balanced = obs_per_entity.nunique() == 1
    
    results = {
        'n_entities': n_entities,
        'n_periods': n_periods,
        'expected_obs': max_obs,
        'actual_obs': actual_obs,
        'missing_obs': max_obs - actual_obs,
        'balance_ratio': balance_ratio,
        'is_balanced': is_balanced,
        'min_obs_per_entity': obs_per_entity.min(),
        'max_obs_per_entity': obs_per_entity.max(),
        'mean_obs_per_entity': obs_per_entity.mean()
    }
    
    return results

# Analyze balanced panel
balance_stats_original = analyze_panel_balance(df_long, 'state', 'year')
print("Original Panel:")
print("=" * 50)
for key, val in balance_stats_original.items():
    print(f"{key:.<30} {val}")

# Analyze unbalanced panel
balance_stats_unbalanced = analyze_panel_balance(df_unbalanced, 'state', 'year')
print("\nUnbalanced Panel:")
print("=" * 50)
for key, val in balance_stats_unbalanced.items():
    print(f"{key:.<30} {val}")

### Visualize Missing Data Pattern

In [ ]:
# Create presence matrix
presence_matrix = df_unbalanced.pivot_table(
    index='state',
    columns='year',
    values='gdp',
    aggfunc='count'
).fillna(0)

# Plot heatmap
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(
    presence_matrix,
    cmap=['lightgray', 'darkblue'],
    cbar_kws={'label': 'Observed'},
    linewidths=0.5,
    ax=ax
)
ax.set_title('Panel Data Coverage (Blue = Observed, Gray = Missing)', fontsize=14)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('State', fontsize=12)
plt.tight_layout()
plt.show()

print("Missing data pattern: Random gaps across states and years")

## 6. Panel Data Validation

Always validate panel structure before analysis.

In [ ]:
def validate_panel_structure(df, entity_col, time_col):
    """
    Validate panel data structure and identify issues.
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
    time_col : str
    
    Returns
    -------
    dict
        Validation results and warnings
    """
    issues = []
    warnings = []
    
    # Check 1: Duplicate observations
    duplicates = df.duplicated(subset=[entity_col, time_col]).sum()
    if duplicates > 0:
        issues.append(f"⚠️  {duplicates} duplicate entity-time combinations found")
    
    # Check 2: Missing entity or time identifiers
    missing_entity = df[entity_col].isna().sum()
    missing_time = df[time_col].isna().sum()
    if missing_entity > 0:
        issues.append(f"⚠️  {missing_entity} missing entity identifiers")
    if missing_time > 0:
        issues.append(f"⚠️  {missing_time} missing time identifiers")
    
    # Check 3: Unbalanced panel
    obs_per_entity = df.groupby(entity_col).size()
    if obs_per_entity.nunique() > 1:
        warnings.append(f"Panel is unbalanced (observations per entity: {obs_per_entity.min()}-{obs_per_entity.max()})")
    
    # Check 4: Gaps in time series
    time_values = sorted(df[time_col].unique())
    if len(time_values) > 1:
        gaps = []
        for i in range(len(time_values) - 1):
            expected_next = time_values[i] + 1
            actual_next = time_values[i + 1]
            if actual_next != expected_next:
                gaps.append((time_values[i], actual_next))
        
        if gaps:
            warnings.append(f"Time gaps detected: {gaps[:3]}..." if len(gaps) > 3 else f"Time gaps: {gaps}")
    
    # Check 5: Very few time periods
    n_periods = df[time_col].nunique()
    if n_periods < 3:
        warnings.append(f"Only {n_periods} time periods (3+ recommended for panel analysis)")
    
    # Summary
    is_valid = len(issues) == 0
    
    return {
        'is_valid': is_valid,
        'issues': issues,
        'warnings': warnings
    }

# Validate original panel
validation = validate_panel_structure(df_long, 'state', 'year')

print("Panel Validation Report")
print("=" * 50)
print(f"Valid structure: {validation['is_valid']}")

if validation['issues']:
    print("\nIssues (must fix):")
    for issue in validation['issues']:
        print(f"  {issue}")
else:
    print("\n✅ No structural issues detected")

if validation['warnings']:
    print("\nWarnings (consider):")
    for warning in validation['warnings']:
        print(f"  {warning}")
else:
    print("\n✅ No warnings")

---

## Exercises

### Exercise 1: Create Panel from Raw Data

**Task:** You receive international trade data in a messy format. Convert it to proper panel structure.

**Expected Output:** MultiIndex DataFrame with country and year as indices

In [ ]:
# Raw trade data (wide format, messy column names)
raw_trade_data = {
    'country': ['USA', 'CHN', 'DEU', 'JPN', 'GBR'],
    'exports_2019': [1643, 2499, 1489, 705, 468],
    'exports_2020': [1431, 2591, 1380, 641, 403],
    'exports_2021': [1754, 3364, 1631, 756, 468],
    'imports_2019': [2568, 2078, 1235, 720, 673],
    'imports_2020': [2407, 2056, 1171, 635, 625],
    'imports_2021': [2935, 2688, 1424, 762, 698],
    'gdp_2019': [21428, 14343, 3861, 5082, 2831],
    'gdp_2020': [20893, 14723, 3806, 5048, 2708],
    'gdp_2021': [22996, 17734, 4223, 4941, 3108]
}

df_trade_wide = pd.DataFrame(raw_trade_data)
print("Raw trade data (wide format):")
print(df_trade_wide)

In [ ]:
# YOUR CODE HERE
# ---------------
# Convert to long format with MultiIndex
# Should have columns: exports, imports, gdp
# Index: country, year

df_trade_panel = None  # Replace with your implementation

In [ ]:
# AUTO-GRADED TESTS
def test_exercise_1():
    assert df_trade_panel is not None, "Don't forget to implement your solution!"
    assert isinstance(df_trade_panel.index, pd.MultiIndex), "Should have MultiIndex"
    assert df_trade_panel.index.names == ['country', 'year'], "Index names should be ['country', 'year']"
    assert len(df_trade_panel) == 15, f"Should have 15 rows (5 countries × 3 years), got {len(df_trade_panel)}"
    assert all(col in df_trade_panel.columns for col in ['exports', 'imports', 'gdp']), "Missing required columns"
    assert df_trade_panel.loc[('USA', 2020), 'exports'] == 1431, "Incorrect value for USA 2020 exports"
    print("✅ Exercise 1 passed!")
    print("\nYour panel structure:")
    print(df_trade_panel.head(10))

# Uncomment to test
# test_exercise_1()

### Exercise 2: Fix Unbalanced Panel

**Task:** Create a balanced panel by filling missing observations with forward fill.

In [ ]:
# YOUR CODE HERE
# ---------------
# Starting with df_unbalanced, create a complete balanced panel
# Use forward fill (ffill) to fill missing values within each state

def balance_panel(df, entity_col, time_col, value_cols):
    """
    Balance panel by filling missing entity-time combinations.
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
    time_col : str
    value_cols : list
        Columns to fill
    
    Returns
    -------
    pd.DataFrame
        Balanced panel
    """
    # TODO: Implement panel balancing
    pass

df_balanced = None  # Replace with your implementation

In [ ]:
# AUTO-GRADED TESTS
def test_exercise_2():
    assert df_balanced is not None, "Don't forget to implement your solution!"
    
    # Check balance
    balance_stats = analyze_panel_balance(df_balanced, 'state', 'year')
    assert balance_stats['is_balanced'], "Panel should be balanced"
    assert len(df_balanced) == 140, f"Should have 140 observations (10 states × 14 years), got {len(df_balanced)}"
    assert df_balanced['gdp'].notna().all(), "Should have no missing GDP values after filling"
    
    print("✅ Exercise 2 passed!")
    print(f"\nBalanced panel: {len(df_balanced)} observations")
    print(f"All entities have {balance_stats['max_obs_per_entity']} observations")

# Uncomment to test
# test_exercise_2()

### Exercise 3: Create Lagged Variables

**Task:** Create lagged variables for panel data (crucial for dynamic panel models).

In [ ]:
# YOUR CODE HERE
# ---------------
# Create 1-period and 2-period lags of GDP for each state
# Must handle entity boundaries correctly

def create_panel_lags(df, entity_col, time_col, var_col, lags):
    """
    Create lagged variables in panel data.
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
    time_col : str
    var_col : str
        Variable to lag
    lags : list
        List of lag periods
    
    Returns
    -------
    pd.DataFrame
        Original data with lagged columns
    """
    # TODO: Implement lagged variable creation
    pass

df_with_lags = None  # Replace with your implementation

In [ ]:
# AUTO-GRADED TESTS
def test_exercise_3():
    assert df_with_lags is not None, "Don't forget to implement your solution!"
    assert 'gdp_lag1' in df_with_lags.columns, "Missing gdp_lag1 column"
    assert 'gdp_lag2' in df_with_lags.columns, "Missing gdp_lag2 column"
    
    # Check lags are correct for one state
    ca_data = df_with_lags[df_with_lags['state'] == 'CA'].sort_values('year').reset_index(drop=True)
    
    # First observation should have NaN lag
    assert pd.isna(ca_data.loc[0, 'gdp_lag1']), "First observation lag1 should be NaN"
    
    # Second observation lag1 should equal first observation gdp
    assert ca_data.loc[1, 'gdp_lag1'] == ca_data.loc[0, 'gdp'], "Lag1 not computed correctly"
    
    # Third observation lag2 should equal first observation gdp
    assert ca_data.loc[2, 'gdp_lag2'] == ca_data.loc[0, 'gdp'], "Lag2 not computed correctly"
    
    print("✅ Exercise 3 passed!")
    print("\nExample lagged data (California):")
    print(ca_data[['year', 'gdp', 'gdp_lag1', 'gdp_lag2']].head(7))

# Uncomment to test
# test_exercise_3()

### Exercise 4: Compute Within and Between Variation

**Task:** Decompose total variation into within-entity and between-entity components.

In [ ]:
# YOUR CODE HERE
# ---------------
# Compute within and between standard deviations

def compute_variation_decomposition(df, entity_col, var_col):
    """
    Decompose variation into within and between components.
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
    var_col : str
    
    Returns
    -------
    dict
        Variation statistics
    """
    # TODO: Compute within and between variation
    # Within: variation around entity means
    # Between: variation of entity means
    pass

variation_stats = None  # Replace with your implementation

In [ ]:
# AUTO-GRADED TESTS
def test_exercise_4():
    assert variation_stats is not None, "Don't forget to implement your solution!"
    assert 'total_sd' in variation_stats, "Missing total_sd"
    assert 'within_sd' in variation_stats, "Missing within_sd"
    assert 'between_sd' in variation_stats, "Missing between_sd"
    
    # Basic sanity checks
    assert variation_stats['total_sd'] > 0, "Total SD should be positive"
    assert variation_stats['within_sd'] > 0, "Within SD should be positive"
    assert variation_stats['between_sd'] > 0, "Between SD should be positive"
    
    print("✅ Exercise 4 passed!")
    print("\nVariation decomposition for GDP:")
    print("=" * 50)
    for key, val in variation_stats.items():
        print(f"{key:.<30} {val:.2f}")
    
    # Interpretation
    within_pct = (variation_stats['within_sd'] / variation_stats['total_sd']) * 100
    between_pct = (variation_stats['between_sd'] / variation_stats['total_sd']) * 100
    print(f"\nWithin variation: {within_pct:.1f}% of total")
    print(f"Between variation: {between_pct:.1f}% of total")

# Uncomment to test
# test_exercise_4()

---

## Summary

### Key Takeaways

1. **Panel Structure:** Entities observed over multiple time periods with MultiIndex

2. **Format Conversion:** Master wide-to-long and long-to-wide transformations

3. **Balanced vs Unbalanced:** Understand implications for estimation

4. **Data Validation:** Always check for duplicates, gaps, and missing patterns

5. **Lagged Variables:** Correctly handle entity boundaries when creating lags

### Common Pitfalls

- Forgetting to sort by entity and time before creating lags
- Not handling entity boundaries (lag of first period should be NaN)
- Assuming balanced panel when data is unbalanced
- Ignoring duplicate entity-time combinations

### What's Next

In the next notebook:
- Explore panel data visually
- Compute within and between statistics
- Understand data variation structure
- Identify suitable panel methods

---

**Next:** `02_exploration.ipynb` - Panel Data Exploration and Visualization

In [ ]:
key_takeaways(["Next:"])